# 03 Environments: why it works on your machine and not on theirs

## 📚 Learning Objectives

By completing this notebook, you will:
- Say exactly which Python interpreter is running any line of code you write, and prove it
- Explain what `PATH`, `sys.executable`, `sys.prefix` and `sys.path` each control, and which one wins
- Read a Jupyter **kernelspec** and connect a notebook to the interpreter it will actually use
- Create, inspect and destroy a virtual environment from the command line
- Write and read a `requirements.txt`, and decide when to pin a version and when not to
- Diagnose `ModuleNotFoundError` for a module that is definitely installed, using a five-rung ladder
  that works every time instead of guessing

## 🎯 219 notebooks, one line of JSON

Every `.ipynb` file carries a small block of metadata naming the kernel it wants:

```json
"kernelspec": { "display_name": "Python 3", "language": "python", "name": "python3" }
```

`"name": "python3"` looks like the safe, obvious, default choice. On this repository it was a bug.

Reading the git history of this repository directly: at commit `8a91c337` there were **421** notebooks
under `Course *`. **219** of them declared `"name": "python3"`, **123** declared no kernelspec at all,
78 declared `ai-diploma` and 1 declared `tfenv`. Commit `7651ada2`, whose message begins *"fix(portability):
every notebook now declares the kernel it w…"*, changed that to the state you can measure yourself later
in this notebook.

The reason `python3` was a bug is that **`python3` is not a program. It is a name that something resolves.**
On the machine that executed this notebook, the kernel named `python3` resolves to
`/Applications/Xcode.app/Contents/Developer/usr/bin/python3` — Python **3.9.6**, the interpreter Apple
ships inside Xcode. It is a perfectly good Python. It is simply not the one this repository was built
against, and the packages it can see are a different, older, incomplete set.

That is the failure this whole lesson is about, and it is worth being precise about why it is nasty.
It did not fail loudly on notebook 1. Part 5 of this notebook interrogates that interpreter directly,
and on the machine that executed this it *does* have numpy, pandas, scikit-learn, matplotlib and torch
— at different versions — because at some point somebody ran `pip install` outside a virtual environment
and those packages landed in a user-level folder this interpreter can see. So the early notebooks ran.
It broke at the first notebook needing `shap`, or `fairlearn`, or `plotly`, which are not there — and by
then nobody connects the error to a kernel choice made weeks earlier. Do not take that on trust: the
cell in Part 5 measures it on *your* machine, and your numbers may differ from the ones stored here.

**A half-working environment is worse than a broken one.** A broken one tells you immediately.

## 🔗 Where this fits

**Builds on:** The earlier lessons of this six-lesson tooling strand — the shell and the filesystem.
Everything below is a real command run in a real shell; if `cd`, `ls` and "absolute versus relative
path" are not yet automatic, go back and do those first.

**Used later in:** Every notebook in all twelve courses, because every one of them runs inside an
interpreter that this lesson teaches you to identify. Most directly: the TensorFlow notebooks in
Course 01 units 3–5 and Course 08, which cannot run in the same environment as the rest of the diploma
and are the case study below. And later in this strand, when you are handed a repository you have never
seen and have to make it run — which is exactly this skill, under time pressure.

**Why this lesson exists at all:** it is the single most common reason a student is blocked, and the
block is invisible. The code is right, the error message is about something else entirely, and no
amount of re-reading the code helps.

---

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Your own machine, as it is right now — its interpreters, its `PATH`, its installed Jupyter kernels.
  Nothing here is simulated; every number below is measured when you run the cell.
- This repository: its 421 course notebooks, its `requirements.txt`, and its two Jupyter kernels.
- A temporary scratch directory that this notebook creates and deletes. Nothing is written into the
  repository, and nothing is committed.

**Outputs:** What you'll see when you run the cells

- A full identification of the interpreter running this notebook, and of every kernel installed for you
- A real virtual environment, built, inspected, proved isolated, and destroyed
- A real `ModuleNotFoundError` for a package that is definitely installed — produced on purpose
- A reusable `diagnose_import()` function and a five-rung diagnostic ladder to keep
- Three broken environments to fix by hand, and one document to write

**No network is required.** Every cell runs offline.

---

## Part 1 — There is no such thing as "Python"

Most people carry a mental model that looks like this: *Python is installed on my computer. I run
`pip install pandas`. Now Python has pandas.*

Every clause of that is wrong, and each wrong clause produces a different bug.

A typical developer machine has between three and ten Python interpreters on it. A macOS laptop can
easily have: the one inside Xcode's command line tools, one from Homebrew, one from python.org, one
per project virtual environment, and one bundled inside some application. They do not share packages.
They do not share versions. Installing into one has no effect on any other.

So the only meaningful question is never *"is pandas installed?"* It is always
**"is pandas installed for the specific interpreter that is about to run my code?"**

Three facts settle that, and you should be able to produce all three in ten seconds:

| Question | Answer lives in | What it tells you |
|---|---|---|
| Which interpreter am I in? | `sys.executable` | The absolute path of the running Python. The only source of truth. |
| Is it a virtual environment? | `sys.prefix` vs `sys.base_prefix` | If they differ, you are in a venv, and `sys.prefix` is its folder. |
| Where will it look for packages? | `sys.path` | An ordered list of directories, searched top to bottom. |

`sys.prefix` and `sys.base_prefix` are not folklore — they are the mechanism specified by
**PEP 405**, which defines a virtual environment as a directory containing a `pyvenv.cfg` file next
to the interpreter. When Python starts, it looks for that file; if it finds one, it sets `sys.prefix`
to that directory and leaves `sys.base_prefix` pointing at the original installation. That single file
is the entire trick. There is nothing else to a venv.

Run the next cell before reading on.

In [1]:
# WHAT: identify, with no ambiguity, the exact interpreter that is running this notebook.
# WHY: every environment problem you will ever have starts by answering this question wrongly.
#      Print these five lines before you debug anything else, forever.
import sys
import os
from pathlib import Path

print("sys.executable  :", sys.executable)      # the absolute path of the running Python
print("sys.version     :", sys.version.split()[0])
print("sys.prefix      :", sys.prefix)          # the environment root
print("sys.base_prefix :", sys.base_prefix)     # the installation this environment was built from

in_venv = sys.prefix != sys.base_prefix
print("in a venv?      :", in_venv, "  (True when the two prefixes differ - PEP 405)")

# WHAT: read the one file that makes a venv a venv.
# WHY: it demystifies the whole thing. A venv is a folder, a copied interpreter, and this text file.
cfg = Path(sys.prefix) / "pyvenv.cfg"
print("\npyvenv.cfg      :", cfg, "(exists:", cfg.exists(), ")")
if cfg.exists():
    for line in cfg.read_text().splitlines():
        print("   |", line)

sys.executable  : /Users/abdullah/Downloads/AI Diploma/.venv/bin/python
sys.version     : 3.14.3
sys.prefix      : /Users/abdullah/Downloads/AI Diploma/.venv
sys.base_prefix : /opt/homebrew/opt/python@3.14/Frameworks/Python.framework/Versions/3.14
in a venv?      : True   (True when the two prefixes differ - PEP 405)

pyvenv.cfg      : /Users/abdullah/Downloads/AI Diploma/.venv/pyvenv.cfg (exists: True )
   | home = /opt/homebrew/opt/python@3.14/bin
   | include-system-site-packages = false
   | version = 3.14.3
   | executable = /opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/bin/python3.14
   | command = /opt/homebrew/opt/python@3.14/bin/python3.14 -m venv /Users/abdullah/Downloads/AI Diploma/.venv


### Reading that output

If `in a venv?` is `True`, `sys.prefix` is the folder you could delete to destroy this environment,
and `pyvenv.cfg` is the file that told Python to use it. `home = …` inside that file points back at
the interpreter the venv was cloned from — that is `sys.base_prefix`.

Notice what is *not* here: nothing about `PATH`, nothing about "activation", no registry, no global
list of installed packages. A virtual environment is a directory. That is the whole idea.

---

## Part 2 — `PATH` decides which Python starts. `sys.executable` reports which one did.

`PATH` is an environment variable holding an ordered list of directories. When you type `python3` at
a shell prompt, the shell walks that list left to right and runs the **first** file called `python3`
it finds. That is the entire algorithm. It explains almost every "but I installed it!" complaint.

Two consequences that catch people:

1. **`pip` has the same problem.** `pip` is also just a file on `PATH`. The `pip` that runs may belong
   to a completely different interpreter than the `python` that runs. This is the classic disaster:
   you `pip install pandas`, it succeeds, and `import pandas` still fails — because they were two
   different Pythons. The fix is a habit, not a command: **never type bare `pip`. Type
   `python -m pip`**, which forces pip to be the one belonging to *that* interpreter.
2. **"Activating" a venv is not magic.** `source .venv/bin/activate` prepends `.venv/bin` to `PATH`
   and sets `VIRTUAL_ENV`. It changes nothing else. Which is why you never actually need it: calling
   `/path/to/.venv/bin/python` directly is exactly equivalent, and is what scripts, CI and Jupyter
   kernels all do.

In [2]:
# WHAT: subprocess settings reused all through this notebook.
# WHY: PYTHON_COLORS=0 stops Python colouring its tracebacks, so the errors we capture on purpose
#      later are stored as plain readable text instead of terminal escape codes.
import subprocess

RUN_ENV = {**os.environ, "PYTHON_COLORS": "0", "NO_COLOR": "1"}

# WHAT: resolve the command names a shell would run, and compare them to the interpreter we are in.
# WHY: this is the "pip installed it but python cannot see it" bug, made visible before it bites you.
import shutil

print("Interpreter actually running this cell:")
print("   sys.executable :", sys.executable)

print("\nWhat the shell would run for each name (first match on PATH):")
for name in ["python", "python3", "pip", "pip3", "jupyter"]:
    found = shutil.which(name)
    print(f"   {name:8} -> {found if found else '(not found on PATH)'}")

# WHAT: ask the shell's python3 which ENVIRONMENT it belongs to, by running it and asking.
# WHY: comparing file paths is not enough. Every venv's bin/python is a link back to the same base
#      binary, so two different environments can resolve to one identical file. sys.prefix is the
#      only comparison that cannot lie, and getting it requires actually running the interpreter.
shell_python = shutil.which("python3")
if shell_python:
    probe = subprocess.run([shell_python, "-c", "import sys; print(sys.prefix)"],
                           capture_output=True, text=True, timeout=60, env=RUN_ENV)
    shell_prefix = probe.stdout.strip()
    print(f"\nThe shell's python3 belongs to environment : {shell_prefix}")
    print(f"This notebook belongs to environment       : {sys.prefix}")
    print("Same environment? ->", shell_prefix == sys.prefix)
else:
    print("\nNo python3 on PATH at all - the shell and this notebook share nothing.")

# WHAT: show PATH as the ordered list it really is, first entries first.
# WHY: order is the whole mechanism - earlier entries win, and that is why activation works.
path_entries = os.environ.get("PATH", "").split(os.pathsep)
print(f"\nPATH has {len(path_entries)} entries. The first 6, in priority order:")
for i, entry in enumerate(path_entries[:6], 1):
    print(f"   {i}. {entry}")

print("\nVIRTUAL_ENV environment variable:", os.environ.get("VIRTUAL_ENV", "(not set)"))

Interpreter actually running this cell:
   sys.executable : /Users/abdullah/Downloads/AI Diploma/.venv/bin/python

What the shell would run for each name (first match on PATH):
   python   -> /Users/abdullah/Downloads/AI Diploma/.venv/bin/python
   python3  -> /Users/abdullah/Downloads/AI Diploma/.venv/bin/python3
   pip      -> /Users/abdullah/Downloads/AI Diploma/.venv/bin/pip
   pip3     -> /Users/abdullah/Downloads/AI Diploma/.venv/bin/pip3
   jupyter  -> /Users/abdullah/Downloads/AI Diploma/.venv/bin/jupyter

The shell's python3 belongs to environment : /Users/abdullah/Downloads/AI Diploma/.venv
This notebook belongs to environment       : /Users/abdullah/Downloads/AI Diploma/.venv
Same environment? -> True

PATH has 27 entries. The first 6, in priority order:
   1. /Users/abdullah/Downloads/AI Diploma/.venv/bin
   2. /opt/homebrew/share/google-cloud-sdk/bin
   3. /Users/abdullah/.antigravity/antigravity/bin
   4. /Users/abdullah/.bun/bin
   5. /Users/abdullah/.local/bin
   6. /us

### Reading that output

Read the last three lines of your own output together. There are two possibilities, and both are
instructive:

- **`Same environment? True`.** A terminal opened here talks to the same environment as this
  notebook, so a `pip install` typed at a shell prompt *would* show up in these cells. Comfortable —
  but it is a property of how this particular session was launched, not a guarantee.
- **`Same environment? False`.** Then the terminal and the notebook are two different worlds, and
  installing in one changes nothing in the other. This is normal, it is not a bug, and it is what
  every "but I just installed it!" question turns out to be about.

Note what we did **not** do: compare file paths. Every venv's `bin/python` is a link back to the same
base interpreter, so resolving symlinks makes two genuinely different environments look identical.
Running the interpreter and asking it for `sys.prefix` is the comparison that cannot lie. Remember
that — it comes back in a moment when we compare Jupyter kernels.

**Habit to build now, and never break:**

```bash
python -m pip install X      # not:  pip install X
python -m venv .venv         # not:  virtualenv .venv
python -m pytest             # not:  pytest
```

`python -m` guarantees the tool and the interpreter are the same environment. Bare command names do not.

---

## Part 3 — `sys.path`: how an import actually finds a module

When you write `import pandas`, Python does four things, in this order:

1. Is `pandas` already in `sys.modules` (imported earlier in this session)? Use that. **Stop.**
2. Is it a built-in module compiled into the interpreter? Use that. **Stop.**
3. Walk `sys.path` **top to bottom**. In each directory, look for `pandas/__init__.py`, then
   `pandas.py`, then a compiled extension. First hit wins. **Stop.**
4. Nothing found -> `ModuleNotFoundError`.

Step 3 is where the interesting failures live, and the phrase that matters is **first hit wins**.

One entry of `sys.path` is the **current working directory**, printed as an empty string `''`. In a
plain script it is entry 0; inside a Jupyter kernel it sits further down — find it in your own output
below. Its exact index does not matter. What matters is that it comes **before** `site-packages`. So a
file named `random.py` in the folder you launched from is found first and imported *instead of*
Python's own `random`. You will see that happen for real later in this notebook.

In [3]:
# WHAT: print sys.path with a label for what each entry is and whether it exists.
# WHY: an import is a top-to-bottom walk of exactly this list. Reading it turns import errors
#      from mysterious into arithmetic.
# The real stdlib directory. NOT sys.base_prefix: on many installs base_prefix is a symlinked
# alias (e.g. .../opt/python@3.14/...) while sys.path holds the real path (.../Cellar/python@3.14/...),
# so a prefix comparison silently fails. os.__file__ is the standard library, by definition.
STDLIB_DIR = str(Path(os.__file__).parent)


def label(entry: str) -> str:
    """Classify one sys.path entry so the list is readable at a glance."""
    if entry == "":
        return "current working directory  <-- searched BEFORE site-packages"
    if entry.endswith(".zip"):
        return "stdlib zip archive (usually absent; harmless)"
    if "site-packages" in entry and entry.startswith(sys.prefix):
        return "THIS ENVIRONMENT's site-packages  <-- pip installs land here"
    if "site-packages" in entry and "Library/Python" in entry:
        return "USER site-packages (outside any venv - shared by every project!)"
    if "site-packages" in entry:
        return "some other site-packages"
    if entry.startswith(STDLIB_DIR):
        return "standard library"
    return "other"

print(f"sys.path has {len(sys.path)} entries, searched in this order:\n")
for i, entry in enumerate(sys.path):
    shown = entry if entry else "'' (empty string)"
    exists = Path(entry).exists() if entry else True
    print(f"{i}. {shown}")
    print(f"     -> {label(entry)}   [exists: {exists}]")

# WHAT: locate a package on disk instead of trusting that it is "installed".
# WHY: "installed" is meaningless; "importable by THIS interpreter, from THIS file" is the fact.
import importlib.util
import importlib.metadata as md

for pkg, dist in [("numpy", "numpy"), ("pandas", "pandas"), ("sklearn", "scikit-learn")]:
    spec = importlib.util.find_spec(pkg)
    try:
        version = md.version(dist)
    except Exception:
        version = "(no metadata)"
    print(f"\n{pkg}: version {version}")
    print(f"   file: {spec.origin if spec else 'NOT IMPORTABLE by this interpreter'}")

sys.path has 5 entries, searched in this order:

0. /opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python314.zip
     -> stdlib zip archive (usually absent; harmless)   [exists: False]
1. /opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14
     -> standard library   [exists: True]
2. /opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/lib-dynload
     -> standard library   [exists: True]
3. '' (empty string)
     -> current working directory  <-- searched BEFORE site-packages   [exists: True]
4. /Users/abdullah/Downloads/AI Diploma/.venv/lib/python3.14/site-packages
     -> THIS ENVIRONMENT's site-packages  <-- pip installs land here   [exists: True]

numpy: version 2.4.4
   file: /Users/abdullah/Downloads/AI Diploma/.venv/lib/python3.14/site-packages/numpy/__init__.py

pandas: version 2.3.3
   file: /Users/abdullah/Downloads/AI Diploma/.venv/lib/python

### Reading that output

Every importable package sits inside one of the directories listed above it. `find_spec()` is the
honest way to ask "can this interpreter import this?" — it answers without actually running the
package's import code, so it is fast and cannot fail for unrelated reasons.

Note the entry labelled **USER site-packages** if you have one. That directory is where
`pip install` writes when it is run by an interpreter with no virtual environment active. It is
shared by *every* project using that interpreter, which is precisely how one project's upgrade
silently breaks another. **PEP 668** exists because of this: it lets a Linux distribution or a
system packager mark its interpreter as "externally managed", which makes pip refuse to install into
it and tell you to make a venv instead. If you have ever seen
`error: externally-managed-environment`, that is PEP 668 protecting you, not obstructing you.

---

## Part 4 — The Jupyter layer: kernelspecs

Everything so far applies to any Python code. Notebooks add one more layer, and it is the layer that
broke this repository.

A notebook does not run Python itself. It sends code to a **kernel** — a separate process — and
displays what comes back. Which process? That is decided by a **kernelspec**: a directory containing
a file called `kernel.json`. Per the Jupyter documentation page *"Making kernels for Jupyter"*,
these directories are searched in several locations, including `{sys.prefix}/share/jupyter/kernels`
for the current environment and, on macOS, `~/Library/Jupyter/kernels` for the current user
(`~/.local/share/jupyter/kernels` on Linux, `%APPDATA%\jupyter\kernels` on Windows).

The important key inside `kernel.json` is `argv` — the documentation describes it as *"a list of
command line arguments used to start the kernel"*, where `{connection_file}` is substituted at launch.
`argv[0]` is therefore **the absolute path of the interpreter your notebook will really use.**

So the chain is:

```
notebook metadata: "kernelspec": {"name": "ai-diploma"}
        |
        v
~/Library/Jupyter/kernels/ai-diploma/kernel.json
        |
        v
argv[0] = /Users/…/AI Diploma/.venv/bin/python      <-- the actual interpreter
```

Three separate places to get it wrong. The next cell reads all of them on **your** machine.

In [4]:
# WHAT: list every Jupyter kernel installed for you, resolve each to its real interpreter, and ask
#       that interpreter directly for its version and its environment.
# WHY: a kernel naming a deleted interpreter is the #1 cause of "the kernel keeps dying", and a kernel
#      pointing at the wrong environment is the #1 cause of "module not found". Both are visible here.
from jupyter_client.kernelspec import KernelSpecManager

# Ask an interpreter to describe itself, exactly as in Part 2. Two facts, one subprocess.
SELF_DESCRIBE = "import sys; print(sys.version.split()[0]); print(sys.prefix)"

specs = KernelSpecManager().get_all_specs()
print(f"{len(specs)} Jupyter kernel(s) installed for this user:\n")

for name in sorted(specs):
    spec = specs[name]["spec"]
    exe = spec["argv"][0]                    # argv[0] IS the interpreter - this is the whole answer
    exists = Path(exe).exists()

    version, prefix = "INTERPRETER MISSING - this kernel cannot start", None
    if exists:
        try:
            # splitlines(), NOT split(): this repository's path contains a space, and splitting
            # on whitespace would silently truncate it to "/Users/abdullah/Downloads/AI".
            out = subprocess.run([exe, "-c", SELF_DESCRIBE], capture_output=True, text=True,
                                 timeout=30, env=RUN_ENV).stdout.splitlines()
            version, prefix = "Python " + out[0].strip(), out[1].strip()
        except Exception as exc:
            version = f"(could not run: {type(exc).__name__})"

    # Compare ENVIRONMENTS, not file paths - see Part 2 for why paths give the wrong answer here.
    marker = "  <== the kernel running this notebook" if prefix == sys.prefix else ""

    print(f"  name         : {name}{marker}")
    print(f"  display_name : {spec['display_name']}")
    print(f"  interpreter  : {exe}")
    print(f"  version      : {version}")
    print(f"  environment  : {prefix if prefix else '-'}")
    print(f"  spec dir     : {specs[name]['resource_dir']}")
    print()

5 Jupyter kernel(s) installed for this user:

  name         : ai-diploma  <== the kernel running this notebook
  display_name : Python 3.14 (.venv AI Diploma)
  interpreter  : /Users/abdullah/Downloads/AI Diploma/.venv/bin/python
  version      : Python 3.14.3
  environment  : /Users/abdullah/Downloads/AI Diploma/.venv
  spec dir     : /Users/abdullah/Library/Jupyter/kernels/ai-diploma

  name         : evalpy3
  display_name : evalpy3
  interpreter  : /private/tmp/claude-501/-Users-abdullah-Downloads-AI-Diploma/ce382213-ac24-41b8-bfc4-374c2459d46e/scratchpad/raterB/eval_env/bin/python
  version      : Python 3.14.3
  environment  : /private/tmp/claude-501/-Users-abdullah-Downloads-AI-Diploma/ce382213-ac24-41b8-bfc4-374c2459d46e/scratchpad/raterB/eval_env
  spec dir     : /Users/abdullah/Library/Jupyter/kernels/evalpy3

  name         : python3
  display_name : Python 3
  interpreter  : /Applications/Xcode.app/Contents/Developer/usr/bin/python3
  version      : Python 3.9.6
  environm

  name         : tfenv
  display_name : Python (tfenv-TF)
  interpreter  : /Users/abdullah/venvs/ai-diploma-tf/bin/python
  version      : Python 3.13.12
  environment  : /Users/abdullah/venvs/ai-diploma-tf
  spec dir     : /Users/abdullah/Library/Jupyter/kernels/tfenv



### 🖐 Your turn — 5 minutes, on your own machine

Do not skip this because the cell already printed. Look at *your* output and answer in writing:

1. How many kernels do you have, and how many distinct interpreters do they point at?
2. Is any interpreter path missing? If so, that kernel is dead — it will appear in your kernel menu
   and fail on the first cell you run.
3. Which of your kernels is `python3`, and which interpreter does *it* resolve to? Write the full
   path down. That is the environment you get every time you accept a notebook's default.
4. Open the `spec dir` for one kernel in a file browser or with `ls`, and read the `kernel.json`
   inside. Confirm for yourself that it is a small, plain, editable text file.
5. Now the important one: **is that interpreter the one your project needs?** Say how you know.

If your answer to 5 is "because the display name says so", you have not answered it. Display names are
free text. Only `argv[0]` is binding.

---

## Part 5 — The case study: one repository, two kernels, on purpose

This repository does something that looks like a mistake and is not. Let's measure it rather than
assert it.

In [5]:
# WHAT: find the repository root by walking upwards, so this notebook works from any directory.
# WHY: a notebook that only runs from one folder is a notebook that breaks the moment it is moved.
#      This is a pattern worth stealing for your own projects.
def find_repo_root(start: Path) -> Path:
    """Walk up from `start` until we find the folder holding requirements.txt and Course 01."""
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "Course 01").is_dir():
            return candidate
    raise FileNotFoundError("Repository root not found above " + str(start))

REPO = find_repo_root(Path.cwd().resolve())
print("Repository root:", REPO)

# WHAT: read the kernelspec declared by every course notebook in this repository.
# WHY: this is the "one line of JSON" from the opening. Now it is a measurement, not a claim.
import json
from collections import Counter

declared = Counter()
for nb_path in sorted(REPO.glob("Course */**/*.ipynb")):
    try:
        meta = json.loads(nb_path.read_text(encoding="utf-8")).get("metadata", {})
    except Exception:
        declared["(unreadable file)"] += 1
        continue
    declared[meta.get("kernelspec", {}).get("name") or "(none declared)"] += 1

total = sum(declared.values())
print(f"\n{total} notebooks under 'Course *'. Kernel declared by each:\n")
for name, count in declared.most_common():
    installed = "installed" if name in specs else "NOT INSTALLED on this machine"
    exe = specs[name]["spec"]["argv"][0] if name in specs else "-"
    print(f"  {count:>4}  {name:<16} [{installed}]")
    print(f"        -> {exe}")

Repository root: /Users/abdullah/Downloads/AI Diploma



421 notebooks under 'Course *'. Kernel declared by each:

   407  ai-diploma       [installed]
        -> /Users/abdullah/Downloads/AI Diploma/.venv/bin/python
    13  tfenv            [installed]
        -> /Users/abdullah/venvs/ai-diploma-tf/bin/python
     1  python3          [installed]
        -> /Applications/Xcode.app/Contents/Developer/usr/bin/python3


### Back to the opening: what the `python3` kernel would actually have given you

One notebook in the count above still declares `python3`. The claim at the top of this lesson was that
this kernel resolves to an interpreter with a *different, older, partial* set of packages — and that
a partial set is worse than none, because it fails late. The next cell measures that instead of
asserting it, by asking that interpreter directly. If you do not have a `python3` kernel, or it points
somewhere else on your machine, the cell says so and skips; your machine is your machine.

In [6]:
# WHAT: interrogate whatever interpreter YOUR "python3" kernel points at, and compare it to ours.
# WHY: this is the failure from the opening of the notebook, measured rather than described.
if "python3" not in specs:
    print("No kernel named 'python3' on this machine - nothing to compare. Skipping.")
else:
    legacy_exe = specs["python3"]["spec"]["argv"][0]
    COMPARE = r"""
import sys, importlib.util as u, importlib.metadata as md, json
res = {"python": sys.version.split()[0], "prefix": sys.prefix, "pkgs": {}}
for mod, dist in [("numpy","numpy"), ("pandas","pandas"), ("sklearn","scikit-learn"),
                  ("matplotlib","matplotlib"), ("torch","torch"),
                  ("shap","shap"), ("fairlearn","fairlearn"), ("plotly","plotly")]:
    if u.find_spec(mod) is None:
        res["pkgs"][mod] = None
    else:
        try:
            res["pkgs"][mod] = md.version(dist)
        except Exception:
            res["pkgs"][mod] = "installed (version unknown)"
print(json.dumps(res))
"""
    run = subprocess.run([legacy_exe, "-c", COMPARE], capture_output=True, text=True,
                         timeout=60, env=RUN_ENV)
    if run.returncode != 0:
        print("That interpreter could not even run the probe:")
        print("   ", run.stderr.strip().splitlines()[-1] if run.stderr.strip() else "(no output)")
    else:
        legacy = json.loads(run.stdout)
        print(f"kernel 'python3' -> {legacy_exe}")
        print(f"   Python {legacy['python']}   (this notebook: {sys.version.split()[0]})")
        print(f"   environment: {legacy['prefix']}\n")
        print(f"   {'package':<12} {'python3 kernel':<22} {'this notebook':<22}")
        missing = 0
        for mod, ver in legacy["pkgs"].items():
            spec_here = importlib.util.find_spec(mod)
            try:
                here_ver = md.version({"sklearn": "scikit-learn"}.get(mod, mod))
            except Exception:
                here_ver = "not installed"
            if spec_here is None:
                here_ver = "not installed"
            shown = ver if ver else "MISSING"
            if ver is None:
                missing += 1
            print(f"   {mod:<12} {shown:<22} {here_ver:<22}")
        total_probed = len(legacy["pkgs"])
        print(f"\n   {missing} of {total_probed} packages are absent from the 'python3' kernel;"
              f" {total_probed - missing} are present.")
        print("   The other packages ARE there - which is exactly the trap. A notebook using only")
        print("   those runs fine, so nobody suspects the kernel until a later notebook fails.")

kernel 'python3' -> /Applications/Xcode.app/Contents/Developer/usr/bin/python3
   Python 3.9.6   (this notebook: 3.14.3)
   environment: /Applications/Xcode.app/Contents/Developer/Library/Frameworks/Python3.framework/Versions/3.9

   package      python3 kernel         this notebook         
   numpy        2.0.2                  2.4.4                 
   pandas       2.3.3                  2.3.3                 
   sklearn      1.6.1                  1.8.0                 
   matplotlib   3.9.4                  3.10.8                
   torch        2.8.0                  2.13.0                
   shap         MISSING                0.52.0                
   fairlearn    MISSING                0.14.0                
   plotly       MISSING                6.9.0                 

   3 of 8 packages are absent from the 'python3' kernel; 5 are present.
   The other packages ARE there - which is exactly the trap. A notebook using only
   those runs fine, so nobody suspects the kernel until

### Why two kernels, and why that is the honest answer

The repository's own `requirements.txt` says it in a comment near the top:

> *TensorFlow is NOT in this file. TensorFlow has no wheel for this venv's Python, so the
> TensorFlow/Keras notebooks (Courses 01 and 08) run in a separate Python 3.13 environment
> registered as the "tfenv" Jupyter kernel.*

Here is what is verifiable offline, on this machine, rather than taken on trust — the next cell
measures it. Be careful about the difference: *"TensorFlow publishes no wheel for Python 3.14"* is a
claim about what is on the package index today, and this notebook has no network, so it cannot check
it. What it **can** check is that the main environment does not have TensorFlow, that the second
environment does, and that they are different Python versions. That is enough to justify the design.

This is worth internalising because you will hit it in industry constantly. A single dependency that
lags the Python release cycle forces the whole project onto an older interpreter — or forces you to
split into two environments. Neither option is nice. The mistake is pretending you can avoid choosing.

In [7]:
# WHAT: ask each of this repository's two kernels, directly, what it can and cannot import.
# WHY: proves the two-environment design from evidence instead of from a comment in a file.
#      find_spec + metadata are used so nothing heavy is actually imported - this stays fast.
PROBE = r"""
import sys, importlib.util as u, importlib.metadata as md, json
out = {"python": sys.version.split()[0], "exe": sys.executable, "packages": {}}
for mod, dist in [("tensorflow","tensorflow"), ("torch","torch"),
                  ("numpy","numpy"), ("sklearn","scikit-learn")]:
    if u.find_spec(mod) is None:
        out["packages"][mod] = None
    else:
        try:
            out["packages"][mod] = md.version(dist)
        except Exception:
            out["packages"][mod] = "installed (version unknown)"
print(json.dumps(out))
"""

for kernel_name in ["ai-diploma", "tfenv"]:
    if kernel_name not in specs:
        print(f"{kernel_name}: not installed on this machine - skipping\n")
        continue
    exe = specs[kernel_name]["spec"]["argv"][0]
    result = subprocess.run([exe, "-c", PROBE], capture_output=True, text=True,
                            timeout=60, env=RUN_ENV)
    info = json.loads(result.stdout)
    print(f"kernel '{kernel_name}'  ->  Python {info['python']}")
    print(f"   {info['exe']}")
    for mod, ver in info["packages"].items():
        print(f"     {mod:<12} {ver if ver else 'NOT AVAILABLE'}")
    print()

kernel 'ai-diploma'  ->  Python 3.14.3
   /Users/abdullah/Downloads/AI Diploma/.venv/bin/python
     tensorflow   NOT AVAILABLE
     torch        2.13.0
     numpy        2.4.4
     sklearn      1.8.0



kernel 'tfenv'  ->  Python 3.13.12
   /Users/abdullah/venvs/ai-diploma-tf/bin/python
     tensorflow   2.21.0
     torch        2.13.0
     numpy        2.5.2
     sklearn      1.9.0



### Reading that output

Two interpreters. Two Python versions. Two independent sets of packages — note that even `numpy`
is at a different version in each, which is exactly what "independent" means.

Now re-read the opening of this notebook with that in mind. A notebook declaring `tfenv` opened
against the `ai-diploma` kernel fails on `import tensorflow`, and the student stares at a TensorFlow
error and starts googling TensorFlow. The problem was never TensorFlow. **The error message names the
symptom; the cause is one layer down, always.**

---

## Part 6 — Build one, from scratch, right now

Enough reading. The next cell creates a real virtual environment. Not a description of one — a real
one, in a temporary directory that this notebook will delete at the end.

The command is one line, and it is the same on macOS, Linux and Windows:

```bash
python -m venv .venv
```

What that does, in full: creates a folder; puts a copy of (or a link to) the interpreter in
`.venv/bin/python` (`.venv\Scripts\python.exe` on Windows); writes `pyvenv.cfg`; creates an empty
`site-packages`; and bootstraps `pip` into it. That is all. There is no system-wide registration,
nothing to uninstall, and deleting the folder deletes the environment completely.

In [8]:
# WHAT: create a scratch working area outside the repository, for everything that follows.
# WHY: a lesson that writes into the repository it is teaching from is a lesson that corrupts it.
#      Every file we make from here on lives under SCRATCH and is deleted in the final cell.
import tempfile
import time

SCRATCH = Path(tempfile.mkdtemp(prefix="envlab_"))
print("Scratch directory:", SCRATCH)
print("Inside the repository?", SCRATCH.is_relative_to(REPO))   # must be False

# WHAT: build a real virtual environment and time it.
# WHY: people avoid venvs because they imagine they are slow or heavyweight. Measure it instead.
VENV = SCRATCH / "demo-venv"
start = time.perf_counter()
subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True, timeout=180, env=RUN_ENV)
elapsed = time.perf_counter() - start

VENV_PY = VENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
print(f"\nCreated {VENV} in {elapsed:.2f} seconds")
print("Its interpreter:", VENV_PY, "(exists:", VENV_PY.exists(), ")")

# WHAT: show what a venv folder actually contains, one level down.
# WHY: demystifies it. There is nothing in here you could not have made by hand.
print("\nTop level of the new environment:")
for item in sorted(VENV.iterdir()):
    kind = "dir " if item.is_dir() else "file"
    print(f"   {kind}  {item.name}")

print("\nIts pyvenv.cfg - the file that makes it an environment:")
for line in (VENV / "pyvenv.cfg").read_text().splitlines():
    print("   |", line)

Scratch directory: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp
Inside the repository? False



Created /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/demo-venv in 0.99 seconds
Its interpreter: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/demo-venv/bin/python (exists: True )

Top level of the new environment:
   file  .gitignore
   dir   bin
   dir   include
   dir   lib
   file  pyvenv.cfg

Its pyvenv.cfg - the file that makes it an environment:
   | home = /opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/bin
   | include-system-site-packages = false
   | version = 3.14.3
   | executable = /opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/bin/python3.14
   | command = /Users/abdullah/Downloads/AI Diploma/.venv/bin/python -m venv /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/demo-venv


In [9]:
# WHAT: ask the NEW interpreter the same five questions we asked our own in Part 1.
# WHY: side by side, the concept stops being abstract. Same machine, same code, different answers.
ASK = r"""
import sys, json
print(json.dumps({
    "executable": sys.executable,
    "version": sys.version.split()[0],
    "prefix": sys.prefix,
    "base_prefix": sys.base_prefix,
    "in_venv": sys.prefix != sys.base_prefix,
    "site_packages": [p for p in sys.path if "site-packages" in p],
}))
"""
fresh = json.loads(subprocess.run([str(VENV_PY), "-c", ASK], capture_output=True, text=True,
                                  timeout=60, env=RUN_ENV).stdout)
mine = {"executable": sys.executable, "version": sys.version.split()[0],
        "prefix": sys.prefix, "base_prefix": sys.base_prefix,
        "in_venv": sys.prefix != sys.base_prefix,
        "site_packages": [p for p in sys.path if "site-packages" in p]}

for field in ["executable", "version", "prefix", "base_prefix", "in_venv"]:
    print(f"{field}")
    print(f"   this notebook : {mine[field]}")
    print(f"   the new venv  : {fresh[field]}")
print("\nsite-packages this notebook can see:")
for p in mine["site_packages"]:
    print("   ", p)
print("site-packages the new venv can see:")
for p in fresh["site_packages"]:
    print("   ", p)

print("\nSame Python VERSION:", mine["version"] == fresh["version"])
print("Same base installation (base_prefix):", mine["base_prefix"] == fresh["base_prefix"])
print("Same PACKAGES:", mine["site_packages"] == fresh["site_packages"])

executable
   this notebook : /Users/abdullah/Downloads/AI Diploma/.venv/bin/python
   the new venv  : /private/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/demo-venv/bin/python
version
   this notebook : 3.14.3
   the new venv  : 3.14.3
prefix
   this notebook : /Users/abdullah/Downloads/AI Diploma/.venv
   the new venv  : /private/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/demo-venv
base_prefix
   this notebook : /opt/homebrew/opt/python@3.14/Frameworks/Python.framework/Versions/3.14
   the new venv  : /opt/homebrew/opt/python@3.14/Frameworks/Python.framework/Versions/3.14
in_venv
   this notebook : True
   the new venv  : True

site-packages this notebook can see:
    /Users/abdullah/Downloads/AI Diploma/.venv/lib/python3.14/site-packages
site-packages the new venv can see:
    /private/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/demo-venv/lib/python3.14/site-packages

Same Python VERSION: True
Same base installation (base_

### Reading that output

Same version. Same base installation. **Different packages.** That is the entire purpose of a virtual
environment stated in three lines: it isolates *dependencies* without duplicating the *interpreter*.

Which sets up the demonstration everyone should see once, deliberately, in a safe place — before they
see it at 2am the night before a deadline.

In [10]:
# WHAT: run the SAME import in two interpreters on the same machine, and capture the real error.
# WHY: this is the exact failure students report as "it says numpy is not installed, but it IS
#      installed". Both statements are true. They are about different interpreters.
CHECK = "import numpy; print('numpy', numpy.__version__, 'from', numpy.__file__)"

for label_, exe in [("this notebook's interpreter", sys.executable),
                    ("the brand-new venv        ", str(VENV_PY))]:
    result = subprocess.run([exe, "-c", CHECK], capture_output=True, text=True,
                            timeout=60, env=RUN_ENV)
    print(f"--- {label_} ---")
    print("    ", exe)
    if result.returncode == 0:
        print("     SUCCESS:", result.stdout.strip())
    else:
        last = [l for l in result.stderr.strip().splitlines() if l.strip()][-1]
        print("     FAILED :", last)
    print()

# WHAT: list what the new environment actually contains, using ITS OWN pip.
# WHY: `python -m pip` is the habit from Part 2. Here it is, doing the job it exists for.
listing = subprocess.run([str(VENV_PY), "-m", "pip", "list", "--disable-pip-version-check"],
                         capture_output=True, text=True, timeout=120, env=RUN_ENV)
print("Everything installed in the new environment:")
for line in listing.stdout.strip().splitlines():
    print("   ", line)

--- this notebook's interpreter ---
     /Users/abdullah/Downloads/AI Diploma/.venv/bin/python
     SUCCESS: numpy 2.4.4 from /Users/abdullah/Downloads/AI Diploma/.venv/lib/python3.14/site-packages/numpy/__init__.py

--- the brand-new venv         ---
     /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/demo-venv/bin/python
     FAILED : ModuleNotFoundError: No module named 'numpy'

Everything installed in the new environment:
    Package Version
    ------- -------
    pip     26.0


### The sentence to remember

> **"numpy is installed" is not a fact about your computer. It is a fact about one interpreter.**

The new environment has essentially nothing in it, and that is correct — it is what "isolated" means.
A fresh venv is the right starting point for every project precisely *because* it is empty: whatever
you then install is a complete, written-down statement of what your project needs.

Which brings us to how you write that statement down.

---

## Part 7 — `requirements.txt`, and the pinning decision

A `requirements.txt` is a plain list of what to install. `python -m pip install -r requirements.txt`
reads it. That much is easy. The interesting part is what you put on each line.

| Form | Meaning | Use it when |
|---|---|---|
| `pandas` | any version, newest available | you genuinely do not care, or you are writing a library |
| `pandas>=2.0` | a floor | you need a feature added in 2.0 |
| `pandas==2.3.3` | exactly this | you need this run to be reproducible |
| `pandas~=2.3.3` | compatible release | you want patches but not feature changes |

`~=` is defined by **PEP 440**: *"for a given release identifier `V.N`, the compatible release clause
is approximately equivalent to the pair of comparison clauses: `>= V.N, == V.*`"*. The PEP's own
example is `~= 1.4.5`, equivalent to `>= 1.4.5, == 1.4.*` — so it accepts 1.4.6 and rejects 1.5.0.

The real decision is a trade-off with no free answer:

- **Pin everything** and your results reproduce, but you stop receiving security fixes and your
  project quietly rots until an upgrade becomes a week of work.
- **Pin nothing** and you always get current libraries, but a notebook that ran in March can fail in
  September because a dependency changed a default, and nothing in your repository records what
  actually worked.

The common professional compromise is two files: a loose `requirements.txt` describing *intent*, and
a generated lock file recording *exactly what was installed* on the machine where it last worked.
The next cell measures this repository's choice and then generates the lock file it does not have.

In [11]:
# WHAT: parse this repository's real requirements.txt and classify every line.
# WHY: reading a dependency file critically is a job skill. Do it here on a file you can inspect.
req_path = REPO / "requirements.txt"
raw_lines = req_path.read_text(encoding="utf-8").splitlines()

requirements = []
for line in raw_lines:
    stripped = line.split("#")[0].strip()      # drop inline comments and blank/comment-only lines
    if stripped:
        requirements.append(stripped)

exact = [r for r in requirements if "==" in r]
bounded = [r for r in requirements if any(op in r for op in (">=", "<=", "~=", "<", ">")) and "==" not in r]
unbounded = [r for r in requirements if r not in exact and r not in bounded]

print(f"{req_path.name}: {len(raw_lines)} lines of text, {len(requirements)} actual requirements")
print(f"   exactly pinned (==)      : {len(exact)}")
print(f"   bounded (>=, ~=, < ...)  : {len(bounded)}")
print(f"   completely unconstrained : {len(unbounded)}")

# WHAT: compare each requirement against the version actually installed right now.
# WHY: the file says what to install; only the interpreter knows what IS installed. They drift.
print("\nrequirement -> version installed in THIS environment (first 12):")
for req in requirements[:12]:
    dist = req.split("[")[0].split("=")[0].split(">")[0].split("<")[0].split("~")[0].strip()
    try:
        installed = md.version(dist)
    except Exception:
        installed = "NOT INSTALLED HERE"
    print(f"   {req:<28} -> {installed}")

print("\nIf a line above says NOT INSTALLED HERE while the tool obviously works, you have met the")
print("third naming problem: the REQUIREMENT name, the IMPORT name and the COMMAND name are three")
print("different things. 'jupyter' is a metapackage - it installs other distributions and may leave")
print("no distribution of its own; the 'jupyter' command comes from jupyter-core. Likewise you install")
print("scikit-learn and import sklearn, install opencv-python and import cv2, install pillow and")
print("import PIL. Never assume the three names match.")

requirements.txt: 68 lines of text, 39 actual requirements
   exactly pinned (==)      : 0
   bounded (>=, ~=, < ...)  : 0
   completely unconstrained : 39

requirement -> version installed in THIS environment (first 12):
   numpy                        -> 2.4.4
   pandas                       -> 2.3.3
   matplotlib                   -> 3.10.8
   seaborn                      -> 0.13.2
   scikit-learn                 -> 1.8.0
   jupyter                      -> NOT INSTALLED HERE
   ipykernel                    -> 7.2.0
   ipywidgets                   -> 8.1.9
   nbformat                     -> 5.10.4
   nbconvert                    -> 7.17.1
   nbclient                     -> 0.11.0
   openpyxl                     -> 3.1.5

If a line above says NOT INSTALLED HERE while the tool obviously works, you have met the
third naming problem: the REQUIREMENT name, the IMPORT name and the COMMAND name are three
different things. 'jupyter' is a metapackage - it installs other distributions and may 

In [12]:
# WHAT: generate a lock file - the exact versions this environment has - into the scratch directory.
# WHY: `pip freeze` is how you answer "what was actually installed the day it worked?".
#      Written to SCRATCH, never into the repository.
frozen = subprocess.run([sys.executable, "-m", "pip", "freeze", "--disable-pip-version-check"],
                        capture_output=True, text=True, timeout=180, env=RUN_ENV)
lock_path = SCRATCH / "requirements.lock.txt"
lock_path.write_text(frozen.stdout, encoding="utf-8")

lock_lines = [l for l in frozen.stdout.splitlines() if l.strip()]
print(f"Wrote {lock_path}")
print(f"   {len(lock_lines)} exactly-pinned lines, generated from {len(requirements)} loose requirements")
print("\nFirst 8 lines of the lock file:")
for line in lock_lines[:8]:
    print("   ", line)

print("\nThe asymmetry that matters:")
print(f"   requirements.txt says WHAT WE WANT   ({len(requirements)} lines, {len(exact)} exact)")
print(f"   the lock file says WHAT WE GOT       ({len(lock_lines)} lines, all exact)")
print("   Direct dependencies are what you chose. The rest were pulled in by those choices.")

Wrote /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/requirements.lock.txt
   289 exactly-pinned lines, generated from 39 loose requirements

First 8 lines of the lock file:
    aiohappyeyeballs==2.6.2
    aiohttp==3.14.1
    aiosignal==1.4.0
    alembic==1.18.4
    annotated-doc==0.0.4
    annotated-types==0.7.0
    anyio==4.13.0
    appnope==0.1.4

The asymmetry that matters:
   requirements.txt says WHAT WE WANT   (39 lines, 0 exact)
   the lock file says WHAT WE GOT       (289 lines, all exact)
   Direct dependencies are what you chose. The rest were pulled in by those choices.


---

## Part 8 — Diagnosing "ModuleNotFoundError" when the module is definitely installed

This is the part to keep. Everything above was so that this makes sense.

When an import fails for a package you know you installed, **do not reinstall it**. Reinstalling is
the reflex, it usually fails again, and it teaches you nothing. Climb this ladder instead. Each rung
is one command, and one of them will be the answer.

| # | Question | How to answer it | If this is the problem |
|---|---|---|---|
| 1 | Which interpreter is running? | `sys.executable` | You are in the wrong environment. Switch. |
| 2 | Which pip installed it? | `python -m pip -V` — it prints its own Python | You installed into a different interpreter. Reinstall with `python -m pip`. |
| 3 | Can this interpreter see it? | `importlib.util.find_spec("pkg")` | Not installed *here*. Install it *here*. |
| 4 | What file is being imported? | `pkg.__file__` | If it is in your project folder, something local is shadowing it. |
| 5 | Which kernel is the notebook using? | `kernel.json` → `argv[0]` | The notebook is pointed at another environment. Fix the kernelspec. |

Rung 4 deserves its own demonstration, because its symptom is the most confusing of all: the import
**succeeds** and then the package behaves as if it were empty.

In [13]:
# WHAT: a reusable diagnostic that answers rungs 1, 3 and 4 of the ladder in one call.
#       (Rung 2 needs a second interpreter to compare against - that is the next cell.
#        Rung 5 is a Jupyter question and was answered back in Part 4.)
# WHY: keep this function. Paste it into any project. It converts a guessing session into a report.
import importlib.util
import importlib.metadata as meta


def diagnose_import(module_name: str, dist_name: str | None = None) -> None:
    """Print everything needed to explain why `import module_name` did or did not work."""
    dist_name = dist_name or module_name
    print(f"=== diagnose_import({module_name!r}) ===")

    # Rung 1 - which interpreter is asking the question?
    print(f"1. interpreter   : {sys.executable}")
    print(f"   python version: {sys.version.split()[0]}")
    print(f"   virtual env   : {sys.prefix if sys.prefix != sys.base_prefix else 'NONE (base install)'}")

    # Rung 3 - can this interpreter find the module at all?
    spec = importlib.util.find_spec(module_name)
    print(f"3. importable    : {spec is not None}")

    # Rung 4 - if so, from exactly which file, and is that file suspiciously local?
    if spec is not None:
        origin = spec.origin or "(namespace package - no single file)"
        print(f"4. loaded from   : {origin}")
        if spec.origin:
            in_cwd = Path(spec.origin).resolve().parent == Path.cwd().resolve()
            print(f"   in current dir: {in_cwd}"
                  f"{'   <-- SHADOWING a real package!' if in_cwd else ''}")
        try:
            print(f"   version       : {meta.version(dist_name)}")
        except Exception:
            print("   version       : (no installed distribution metadata - not pip-installed?)")
    else:
        print("   -> not on sys.path for this interpreter. Install it HERE:")
        print(f"      {sys.executable} -m pip install {dist_name}")
    print()


diagnose_import("numpy")
diagnose_import("tensorflow")   # expected to be absent in the main environment - see Part 5

=== diagnose_import('numpy') ===
1. interpreter   : /Users/abdullah/Downloads/AI Diploma/.venv/bin/python
   python version: 3.14.3
   virtual env   : /Users/abdullah/Downloads/AI Diploma/.venv
3. importable    : True
4. loaded from   : /Users/abdullah/Downloads/AI Diploma/.venv/lib/python3.14/site-packages/numpy/__init__.py
   in current dir: False
   version       : 2.4.4

=== diagnose_import('tensorflow') ===
1. interpreter   : /Users/abdullah/Downloads/AI Diploma/.venv/bin/python
   python version: 3.14.3
   virtual env   : /Users/abdullah/Downloads/AI Diploma/.venv
3. importable    : False
   -> not on sys.path for this interpreter. Install it HERE:
      /Users/abdullah/Downloads/AI Diploma/.venv/bin/python -m pip install tensorflow



In [14]:
# WHAT: rung 2 - prove that `pip` and `python -m pip` can be different programs.
# WHY: this single confusion produces more wasted hours than any other item in this lesson.
print("The pip belonging to THIS interpreter (`python -m pip -V`):")
own = subprocess.run([sys.executable, "-m", "pip", "-V", "--disable-pip-version-check"],
                     capture_output=True, text=True, timeout=60, env=RUN_ENV)
print("   ", own.stdout.strip())

print("\nThe pip belonging to the scratch venv:")
other = subprocess.run([str(VENV_PY), "-m", "pip", "-V", "--disable-pip-version-check"],
                       capture_output=True, text=True, timeout=60, env=RUN_ENV)
print("   ", other.stdout.strip())

print("\nWhat a bare `pip` would run, from this notebook's PATH:")
bare = shutil.which("pip")
print("   ", bare if bare else "(no pip on PATH at all)")
print("\nRead the tail of each line above: pip always tells you which Python it belongs to.")
print("If that Python is not the one you are running, nothing you install with it will be visible.")

The pip belonging to THIS interpreter (`python -m pip -V`):
    pip 26.0.1 from /Users/abdullah/Downloads/AI Diploma/.venv/lib/python3.14/site-packages/pip (python 3.14)

The pip belonging to the scratch venv:
    pip 26.0 from /private/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/demo-venv/lib/python3.14/site-packages/pip (python 3.14)

What a bare `pip` would run, from this notebook's PATH:
    /Users/abdullah/Downloads/AI Diploma/.venv/bin/pip

Read the tail of each line above: pip always tells you which Python it belongs to.
If that Python is not the one you are running, nothing you install with it will be visible.


### Rung 4: shadowing, the one that looks like a broken library

Recall step 3 of the import algorithm: **walk `sys.path` top to bottom, first hit wins**, and the first
entry is usually the current directory. So if you save a file called `pandas.py` in your project — a
completely reasonable thing to name a scratch script — then `import pandas` finds *your* file first.

The import succeeds. `pd.DataFrame` then does not exist, and you get
`AttributeError: module 'pandas' has no attribute 'DataFrame'` — an error that appears to say pandas
is broken. It is not. You shadowed it.

The next cell does this on purpose, in the scratch directory, and captures the real error.

In [15]:
# WHAT: create a file that shadows a real package, then import it, and capture the true error.
# WHY: seeing this once, deliberately, saves an evening later. It happens in the scratch dir only.
shadow_dir = SCRATCH / "shadow_demo"
shadow_dir.mkdir(exist_ok=True)
(shadow_dir / "pandas.py").write_text('SOMETHING = "this is my own scratch file, not pandas"\n',
                                      encoding="utf-8")

USE_PANDAS = "import pandas as pd; print('imported from:', pd.__file__); pd.DataFrame({'a': [1]})"

for where, cwd in [("from a clean directory", SCRATCH), ("from the directory containing pandas.py", shadow_dir)]:
    result = subprocess.run([sys.executable, "-c", USE_PANDAS], cwd=str(cwd),
                            capture_output=True, text=True, timeout=60, env=RUN_ENV)
    print(f"--- running the SAME line {where} ---")
    for line in result.stdout.strip().splitlines():
        print("     ", line)
    if result.returncode != 0:
        tail = [l for l in result.stderr.strip().splitlines() if l.strip()]
        print("      ERROR:", tail[-1])
        print("      (note: the error blames pandas. pandas is fine.)")
    print()

print("Same interpreter. Same one line of code. Different directory. Different result.")
print("The only thing that changed was the current-working-directory entry of sys.path -")
print("the '' you found in Part 3 - which is searched before site-packages.")

--- running the SAME line from a clean directory ---
      imported from: /Users/abdullah/Downloads/AI Diploma/.venv/lib/python3.14/site-packages/pandas/__init__.py

--- running the SAME line from the directory containing pandas.py ---
      imported from: /private/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/shadow_demo/pandas.py
      ERROR: AttributeError: module 'pandas' has no attribute 'DataFrame'
      (note: the error blames pandas. pandas is fine.)

Same interpreter. Same one line of code. Different directory. Different result.
The only thing that changed was the current-working-directory entry of sys.path -
the '' you found in Part 3 - which is searched before site-packages.


---

## 🖐 Your turn — the triage drill (about 30 minutes, hands on keyboard)

Below, the notebook builds three broken environments in the scratch directory. Each one is a real
failure taken from the list above. **No fix is written down anywhere in this notebook.** The setup
code is visible on purpose — reading it tells you what was *planted*, which is not the same as knowing
why it breaks, which rung of the ladder catches it, or how to prove your explanation is right. Those
three things are the assessed work, and they are what transfers to a bug nobody planted for you.

For each case, in a terminal or in a new cell:

1. Reproduce the failure. Run the command the cell prints and read the actual error.
2. Climb the ladder from Part 8. Write down what each rung answers — all of them, in order, even
   when you think you already know. Use `diagnose_import()`.
3. Name the rung that broke, in one sentence, using the vocabulary of this lesson
   (`sys.executable`, `sys.path`, `site-packages`, `argv[0]`, shadowing).
4. Write the **one command** that fixes it. One. If your fix needs three commands, you have not
   found the cause yet.
5. Run your fix and prove it worked.

Hand your instructor the four written answers, not the fixed folder. The written diagnosis is the
thing being assessed, because it is the thing that transfers to a bug you have not seen before.

In [16]:
# WHAT: build three deliberately broken situations for you to diagnose by hand.
# WHY: reading a diagnostic ladder teaches nothing. Running it against a real failure does.
#      Re-run this cell at any time to rebuild the drill from scratch.
DRILL = SCRATCH / "drill"


def build_drill() -> Path:
    """Create three broken cases under SCRATCH/drill and print the task for each."""
    if DRILL.exists():
        shutil.rmtree(DRILL)
    DRILL.mkdir(parents=True)

    # Case A: correct code, wrong interpreter.
    case_a = DRILL / "case_A"
    case_a.mkdir()
    (case_a / "analyse.py").write_text(
        "import pandas as pd\n"
        "df = pd.DataFrame({'city': ['Riyadh', 'Jeddah'], 'sites': [3, 2]})\n"
        "print(df)\n", encoding="utf-8")

    # Case B: a local file with an unlucky name.
    case_b = DRILL / "case_B"
    case_b.mkdir()
    (case_b / "json.py").write_text(
        "# A helper someone wrote to load the project's config file.\n"
        "def load_config(path):\n"
        "    return {'ok': True}\n", encoding="utf-8")
    (case_b / "report.py").write_text(
        "import json\n"
        "print(json.dumps({'total': 42}))\n", encoding="utf-8")

    # Case C: a requirements file that does not match the environment.
    case_c = DRILL / "case_C"
    case_c.mkdir()
    (case_c / "requirements.txt").write_text(
        "numpy==1.11.0\npandas\nscikit-learn\n", encoding="utf-8")
    (case_c / "train.py").write_text(
        "import numpy as np\n"
        "assert np.__version__.startswith('1.11'), (\n"
        "    'This project pins numpy 1.11 but got ' + np.__version__)\n"
        "print('ok')\n", encoding="utf-8")

    return DRILL


build_drill()

print("Drill built at:", DRILL, "\n")
print("CASE A - correct code, nothing wrong with it, and it fails.")
print(f"   run:  {VENV_PY} {DRILL / 'case_A' / 'analyse.py'}")
print("   then: run the SAME file with this notebook's interpreter and compare.\n")

print("CASE B - a standard-library import that returns the wrong thing.")
print(f"   run:  cd {DRILL / 'case_B'} && {sys.executable} report.py")
print("   then: run the same file from any other directory and compare.\n")

print("CASE C - requirements.txt and reality disagree.")
print(f"   read: {DRILL / 'case_C' / 'requirements.txt'}")
print(f"   run:  {sys.executable} {DRILL / 'case_C' / 'train.py'}")
print("   Do NOT install anything for this one - the network is not the point.")
print("   Explain in writing what the pin claims, what is actually installed, why the pin exists,")
print("   and what you would do about it if this were a repository you had just been handed.\n")

print("Ladder to use, from Part 8: sys.executable -> python -m pip -V -> find_spec -> __file__ -> argv[0]")

Drill built at: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/drill 

CASE A - correct code, nothing wrong with it, and it fails.
   run:  /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/demo-venv/bin/python /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/drill/case_A/analyse.py
   then: run the SAME file with this notebook's interpreter and compare.

CASE B - a standard-library import that returns the wrong thing.
   run:  cd /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/drill/case_B && /Users/abdullah/Downloads/AI Diploma/.venv/bin/python report.py
   then: run the same file from any other directory and compare.

CASE C - requirements.txt and reality disagree.
   read: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/drill/case_C/requirements.txt
   run:  /Users/abdullah/Downloads/AI Diploma/.venv/bin/python /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp/drill/case_C/train.py
   

### 🖐 Your turn — the deliverable: write `ENVIRONMENT.md`

This is the thing this whole strand exists for. A graduate who can be handed a repository and be
useful is a graduate who can write this document, and who notices when a repository is missing it.

Create a file called `ENVIRONMENT.md` for **this** repository. Do not copy the existing setup guide —
write it from what you measured in this notebook. It must contain, and each item must be something
you personally verified today:

1. **The interpreter.** Exact Python version, and the exact command to create the environment.
2. **The install command.** Copy-pasteable, working, in order, starting from a fresh clone.
3. **The kernel.** The command that registers the Jupyter kernel, and the `name` that notebooks must
   declare. State the consequence of getting it wrong.
4. **The second environment.** Why `tfenv` exists, which notebooks need it, and how to tell — from
   the notebook file itself — which one a given notebook wants.
5. **A verification block.** Three commands a newcomer runs to confirm the setup worked, with the
   expected output of each. If they run all three and all three pass, they are ready. This is the
   part that turns a README into a contract.
6. **A known-problems section.** At least three failures from this lesson, each with its symptom,
   its cause in one line, and its fix.

Two rules that make this real:

- **Test it on someone else.** Hand it to a classmate who has not set the project up, watch them
  follow it *literally*, and change every step where they hesitated. Do not help them. The hesitation
  is the finding.
- **Every command must be one you actually ran.** If you did not run it, it does not go in the file.

Save it outside this repository — your own notes folder or your own repo. Do not add it here.

---

## Cleaning up

A lesson that leaves debris on your machine is a bad lesson. The next cell deletes the scratch
directory and everything in it — the venv, the lock file, the shadow demo, the drill.

**Run this only when you have finished the drill.** If you delete it too early, re-run three cells
in order: the one that creates `SCRATCH`, the one that builds `demo-venv`, and the one that calls
`build_drill()`. Everything comes back.

In [17]:
# WHAT: remove every file this notebook created, and prove that nothing was left behind.
# WHY: deleting a virtual environment is deleting a folder - nothing is registered anywhere else.
#      This is also the proof that the notebook never wrote into the repository it teaches from.
print("Everything this notebook created lives under:", SCRATCH)
print("Was that path inside the repository?", SCRATCH.is_relative_to(REPO))

created = sorted(p.relative_to(SCRATCH) for p in SCRATCH.rglob("*") if p.is_file())
print(f"Files created: {len(created)} (the venv accounts for almost all of them)")

shutil.rmtree(SCRATCH, ignore_errors=True)
print("\nRemoved. Scratch directory still exists?", SCRATCH.exists())

# WHAT: independently confirm nothing named like our scratch area ended up in the repository.
# WHY: "I cleaned up" is a claim; a search of the repository is evidence.
strays = list(REPO.rglob("envlab_*")) + list(REPO.rglob("demo-venv"))
print("Stray files left inside the repository:", len(strays))
print("A virtual environment is a folder. Deleting the folder deletes the environment. That is all.")

Everything this notebook created lives under: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/envlab_z66f5vhp
Was that path inside the repository? False
Files created: 896 (the venv accounts for almost all of them)

Removed. Scratch directory still exists? False


Stray files left inside the repository: 0
A virtual environment is a folder. Deleting the folder deletes the environment. That is all.


## 💬 Discuss

Argue these in a group. Each one has a defensible answer on both sides, and each one is a decision
you will genuinely have to make on a real team.

1. This repository's `requirements.txt` pins **nothing** — every one of its requirement lines is
   unconstrained (you counted them in Part 7). **Defend that choice**, then **attack it.** Then decide
   what you would actually do for a repository that students must be able to run in three years'
   time, and say what it costs.
2. The two-kernel split exists because one dependency lags the Python release cycle. The alternatives
   were: hold the entire diploma back a Python version, drop the affected notebooks, or run two
   environments. **Which would you have chosen, and what is the cost you are accepting?** Every option
   here is bad; the skill is naming which badness you prefer.
3. Something committed 219 notebooks pointing at the wrong kernel, and it was not caught for a long
   time. **Design the check that would have caught it on day one.** What exactly does it compare, what
   does it do when it fails, and when does it run? Be concrete enough that someone could implement it.
4. A teammate says: *"virtual environments are pointless, just install what you need."* Using only
   evidence you produced in this notebook, make the shortest possible case that convinces them. Now
   argue the other side — when **is** a venv genuine overhead not worth it?
5. Everything in this lesson could be replaced by a container: one image, one Python, no PATH, no
   kernels. **Why does this course teach venvs instead?** And when a project does move to containers,
   which of today's ideas survive unchanged and which stop mattering?

---

## ⚠️ Where this breaks

**Honesty about this lesson's evidence base, first.** The rest of this diploma cites outcome research
for its teaching choices. This tooling strand cannot, and you should know that. There is no controlled
study showing that teaching virtual environments improves a graduate's employment outcomes, and it
would be dishonest to imply otherwise. The strand is here on a **prerequisite argument**, not a proven
one: peer review of real work, feedback written inside real artefacts, and being handed a repository
and made useful all *presuppose* that you can run someone else's code. Every one of those is
undeliverable without this. That is a strong argument. It is not the same kind of evidence as an
effect size, and it is not presented as one.

**Now the technical limits of what you just learned.**

- **`venv` is the floor, not the ceiling.** Real teams increasingly use `conda`/`mamba` (which manage
  non-Python dependencies like CUDA that pip cannot), `poetry` and `uv` (which produce true lock files
  with a resolver), or containers (which pin the operating system too). The mental model transfers
  directly; the commands do not. Do not assume `python -m venv` is what your first employer uses.
- **`pip freeze` is not a real lock file.** It records what is installed *on this machine, for this
  operating system and CPU*. It does not record hashes, it does not distinguish the packages you asked
  for from the ones pulled in underneath them, and it can produce a file that will not install on
  Linux after being generated on a Mac. Proper lock files (`poetry.lock`, `uv.lock`,
  `pip-compile`'s output) solve this. `pip freeze` is the honest minimum, not the professional answer.
- **A venv does not isolate the operating system.** It shares the system's compilers, GPU drivers,
  CUDA, BLAS and system libraries. "It works in my venv and not in yours" is still entirely possible
  and, when it happens, is usually one of those. Containers exist for exactly this gap.
- **Two kernels is a workaround, not a design.** It carries a permanent cost: results from the two
  environments are not strictly comparable, because they run different Python and different library
  versions. If you ever compare a timing or a metric across the two kernels in this diploma, say so
  out loud in your report.
- **Everything measured here is one machine on one day.** Your kernel list, your `PATH`, your package
  versions and even which of these commands exist will differ from a classmate's. That is not a flaw
  in the lesson — it is the fact the lesson is about. Re-measure on every machine; never carry
  yesterday's answer forward.
- **This lesson ran offline on purpose, which hides the hardest failures.** Proxies, corporate TLS
  interception, private package indices, expired certificates and rate limits produce install errors
  far more baffling than anything demonstrated here. You will meet them at work; you did not meet
  them today.

---

## 📚 References

Each of the following was retrieved and checked while writing this notebook.

1. Meyer, C. (2011). *PEP 405 — Python Virtual Environments*. Status: Final, Standards Track,
   created 13 June 2011. Specifies `pyvenv.cfg`, `sys.prefix` and `sys.base_prefix`.
   <https://peps.python.org/pep-0405/>
2. Coghlan, A., & Stufft, D. (2013). *PEP 440 — Version Identification and Dependency Specification*.
   Status: Final, Standards Track, created 18 March 2013. Source of the `~=` compatible-release
   definition quoted in Part 7. <https://peps.python.org/pep-0440/>
3. Thomas, G., Klose, M., Laíns, F., Stufft, D., Chung, T., Rivera, S., Hashman, E., & Gedam, P.
   (2021). *PEP 668 — Marking Python base environments as "externally managed"*. Status: Final,
   Standards Track, created 18 May 2021. The reason `pip install` is refused on some system Pythons.
   <https://peps.python.org/pep-0668/>
4. Jupyter Project. *Making kernels for Jupyter* (jupyter_client documentation). Source of the
   kernelspec search-path table and the description of the `argv` key.
   <https://jupyter-client.readthedocs.io/en/stable/kernels.html>
5. This repository: `requirements.txt` (the TensorFlow note quoted in Part 5) and
   `docs/SETUP_GUIDE.md`. Read them both; you are about to write a better one.
6. This repository's git history, read directly with `git ls-tree` and `git show`: commits
   `8a91c337` (421 notebooks; 219 declaring `python3`, 123 declaring nothing) and `7651ada2`
   ("every notebook now declares the kernel it w…"). Verify the numbers yourself rather than taking
   them from this notebook — `git ls-tree -r --name-only <commit>` and `git show <commit>:<file>` are
   the two commands, and they need no network.